In [1]:
%pip install nltk


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import ssl

In [3]:
ssl._create_default_https_context = ssl._create_unverified_context
# Ensure required lexical assets are downloaded
nltk.download('vader_lexicon')

# Configuration settings
INPUT_CSV_PATH = "../data/processed_allergy_hazard_dataset.csv"
OUTPUT_ENRICHED_PATH = "./enriched_allergy_hazard_dataset.csv"

# Set visualization styles
sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 12, 'axes.labelsize': 14, 'axes.titlesize': 16})

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/iriskronfeld/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [4]:
df = pd.read_csv(INPUT_CSV_PATH)
print(f"Loaded database shape: {df.shape}")

Loaded database shape: (7500, 9)


SECTION 1: FEATURE ENGINEERING

In [5]:
print("\n--- Executing Feature Engineering Pipeline ---")

# Feature 1: Medical Lexicon Density
medical_terms = {'hospital', 'er', 'doctor', 'ambulance', 'hives', 'swelled', 'vomit', 'breathing', 'emergency', 'epipen'}
def calculate_medical_density(text):
    if not isinstance(text, str):
        return 0.0
    tokens = text.lower().split()
    if len(tokens) == 0:
        return 0.0
    match_count = sum(1 for token in tokens if token in medical_terms)
    return float(match_count / len(tokens))

print("Generating Feature 1: medical_lexicon_density...")
df['medical_lexicon_density'] = df['text'].apply(calculate_medical_density)

# Feature 2: VADER Negative Sentiment Intensity
print("Generating Feature 2: vader_neg_intensity...")
vader_analyzer = SentimentIntensityAnalyzer()
def extract_vader_neg(text):
    if not isinstance(text, str):
        return 0.0
    return float(vader_analyzer.polarity_scores(text)['neg'])

df['vader_neg_intensity'] = df['text'].apply(extract_vader_neg)

# Feature 3: Negation Window Proximity Flag
negation_tokens = {'no', 'not', 'didnt', 'didnot', 'never', 'wasnt', 'without', 'free'}
allergy_tokens = {'allergy', 'allergic', 'celiac', 'nuts', 'peanuts', 'gluten', 'dairy'}

def check_negation_window(text):
    if not isinstance(text, str):
        return 0
    # Clean text to alphanumeric tokens for window evaluation
    tokens = re.sub(r'[^a-zA-Z\s]', '', text.lower()).split()
    for i in range(len(tokens)):
        if tokens[i] in negation_tokens:
            # Inspect subsequent 3 tokens for target safety terms
            window = tokens[i+1 : i+4]
            if any(t in allergy_tokens for t in window):
                return 1
    return 0

print("Generating Feature 3: negation_window_flag...")
df['negation_window_flag'] = df['text'].apply(check_negation_window)

# Save enriched dataset to disk
df.to_csv(OUTPUT_ENRICHED_PATH, index=False, encoding='utf-8')
print(f"Enriched database saved to: {OUTPUT_ENRICHED_PATH}")


--- Executing Feature Engineering Pipeline ---
Generating Feature 1: medical_lexicon_density...
Generating Feature 2: vader_neg_intensity...
Generating Feature 3: negation_window_flag...
Enriched database saved to: ./enriched_allergy_hazard_dataset.csv


SECTION 2: EXPLORATORY DATA VISUALIZATION

In [6]:
print("\n--- Generating Visualization Figures ---")

# Figure 1: Overlapping Kernel Density Estimate for Word Count
plt.figure(figsize=(10, 6))
sns.kdeplot(data=df, x="word_count", hue="is_hazard", common_norm=False, fill=True, palette="muted", alpha=0.5)
plt.title("Graph 1: Distribution of Review Word Counts by Hazard Status")
plt.xlabel("Word Count (Tokens)")
plt.ylabel("Density")
plt.xlim(0, 500)
plt.tight_layout()
plt.savefig("graph1_word_count_distribution.png", dpi=300)
plt.close()
print("Saved: graph1_word_count_distribution.png")


--- Generating Visualization Figures ---
Saved: graph1_word_count_distribution.png


In [7]:
# Figure 2: Bivariate Factorized Bar Plot for Exclamations vs Ratings
plt.figure(figsize=(10, 6))
sns.barplot(data=df, x="stars", y="exclamation_count", hue="is_hazard", palette="deep", errorbar=('ci', 95))
plt.title("Graph 2: Urgency Expression (Exclamation Marks) vs. Customer Ratings")
plt.xlabel("Star Rating")
plt.ylabel("Mean Exclamation Count")
plt.legend(title="Is Hazard")
plt.tight_layout()
plt.savefig("graph2_exclamation_vs_stars.png", dpi=300)
plt.close()
print("Saved: graph2_exclamation_vs_stars.png")

Saved: graph2_exclamation_vs_stars.png


In [8]:
# Figure 3: Feature Interaction (Medical Density vs Negative Intensity)
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x="is_hazard", y="vader_neg_intensity", palette="pastel")
plt.title("Graph 3: Distribution of Negative Sentiment Intensity Across Classes")
plt.xlabel("Is Hazard Label")
plt.ylabel("VADER Negative Valence Score")
plt.tight_layout()
plt.savefig("graph3_sentiment_boxplot.png", dpi=300)
plt.close()
print("Saved: graph3_sentiment_boxplot.png")

print("\n--- Summary Statistics For Review ---")
print(df[['medical_lexicon_density', 'vader_neg_intensity', 'negation_window_flag']].describe())

Saved: graph3_sentiment_boxplot.png

--- Summary Statistics For Review ---
       medical_lexicon_density  vader_neg_intensity  negation_window_flag
count              7500.000000          7500.000000           7500.000000
mean                  0.000177             0.056640              0.006933
std                   0.001853             0.066031              0.082983
min                   0.000000             0.000000              0.000000
25%                   0.000000             0.000000              0.000000
50%                   0.000000             0.038000              0.000000
75%                   0.000000             0.085000              0.000000
max                   0.050000             0.509000              1.000000


/var/folders/fs/dkdlms8103b18vbbt074brch0000gn/T/ipykernel_16424/1188284523.py:3: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df, x="is_hazard", y="vader_neg_intensity", palette="pastel")
